In [ ]:
import os
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

sys.path.append("../..")

from main import show_layer_color_histogram

In [ ]:
folder_0610 = "/path/to/my/folder"  

In [ ]:
result_dataframes = []
subfolder_0610 = [f.path for f in os.scandir(folder_0610) if f.is_dir()]
for subfolder_0610 in subfolder_0610:
    field_name = os.path.basename(subfolder_0610)
    files = [f for f in os.listdir(subfolder_0610) if f.endswith(".csv")]
    try:
        df = pd.read_csv(os.path.join(subfolder_0610, files[0]))
        df['field_name'] = field_name
        df['date'] = '0610'
        result_dataframes.append(df)
    except Exception as e:
        print(f"Error processing {subfolder_0610}: {e}")
        continue
df_0610 = pd.concat(result_dataframes, ignore_index=True)

In [ ]:
df_concatenated = df_0610

In [ ]:
show_layer_color_histogram(
    df_concatenated,
    intensity="center_5pixel"
    )

In [ ]:
layer_2_df = pd.read_csv(
    "../output/20250601-134221/A1_178_concatenated.csv"
)
layer_2_df

In [ ]:
layer_2_log_intensity = np.log(layer_2_df["c3_intensity_center_5pixel"] + 1)
layer_2_log_intensity.hist(bins=100)

In [ ]:
from scipy.optimize import curve_fit


def gaussian(x, amp, mean, sigma):
    return amp * np.exp(-((x - mean) ** 2) / (2 * sigma ** 2))

hist, bin_edges = np.histogram(layer_2_log_intensity, bins=100, density=True)
bin_centers = (bin_edges[:-1] + bin_edges[1:]) / 2

popt, _ = curve_fit(gaussian, bin_centers, hist, p0=[1, np.mean(layer_2_log_intensity), np.std(layer_2_log_intensity)])

mean, sigma = popt[1], popt[2]
print(f"Mean: {mean}, Sigma: {sigma}")
plt.figure(figsize=(10, 6))
plt.hist(layer_2_log_intensity, bins=100, density=True, alpha=0.6, label="Histogram")
plt.plot(bin_centers, gaussian(bin_centers, *popt), label="Gaussian Fit", color="red")
plt.xlabel("Log Intensity")
plt.ylabel("Density")
plt.title("Histogram and Gaussian Fit")
plt.legend()
plt.show()

lower_bound = mean - 3 * sigma
upper_bound = mean + 3 * sigma
lower_bound_exp = np.exp(lower_bound)
upper_bound_exp = np.exp(upper_bound)

print(f"3σ Range (Log Intensity): [{lower_bound}, {upper_bound}]")
print(f"3σ Range (Exponential): [{lower_bound_exp}, {upper_bound_exp}]")

In [ ]:
df_layer_2_blue = df_concatenated[(df_concatenated["layer_id"] == "Layer 2")]
layer_2_blue = df_layer_2_blue["c3_intensity_center_5pixel"].values
df_layer_2_blue["layer_2_group"] = (layer_2_blue > lower_bound_exp) & (layer_2_blue < upper_bound_exp)
df_layer_2_blue["layer_2_group"]

In [ ]:
df_all = df_concatenated.copy()

In [ ]:
df_all["layer_2_blue_intensity_class"] = np.array([[i]*5 for i in df_layer_2_blue["layer_2_group"].values]).flatten()

In [ ]:
df_intensity_class_1 = df_all[df_all["layer_2_blue_intensity_class"] == 1]
show_layer_color_histogram(
    df_intensity_class_1,
    intensity="center_5pixel"
)

In [ ]:
df_layer_4_blue = df_intensity_class_1[(df_intensity_class_1["layer_id"] == "Layer 4")]
layer_4_blue = df_layer_4_blue["c3_intensity_center_5pixel"].values
df_layer_4_blue["layer_4_group"] = (layer_4_blue > 800) & (layer_4_blue < 2000)
df_intensity_class_1["layer_4_blue_intensity_class"] = np.array([[i]*5 for i in df_layer_4_blue["layer_4_group"].values]).flatten()
df_filtered_2 = df_intensity_class_1[df_intensity_class_1["layer_4_blue_intensity_class"] == 1]
show_layer_color_histogram(
    df_filtered_2,
    intensity="center_5pixel"
)

In [ ]:
df_intensity_class_1.to_csv("../output/0610_layer2_filt.csv", index=False)
df_filtered_2.to_csv("../output/0610_layer2_filt_800-2000.csv", index=False)